In [71]:
import numpy as np
import pandas as pd

In [72]:
"""Load the pickle file"""
df=pd.read_pickle(r"C:\Users\lenovo\Desktop\Projects\dataset\titanic_preprocessed.pkl")
df['Sex']=df['Sex'].map({'male':0,'female':1})
df = pd.get_dummies(df, columns=['Embarked'],dtype=float)
X=df[['Pclass','Sex','Age','SibSp','Parch','Fare','Embarked_S','Embarked_C','Embarked_Q']].values
Y=df['Survived'].values
print(Y[0])

0


In [85]:
'''Train test split'''
from sklearn.model_selection import train_test_split as tts
x_train,x_test,y_train,y_test=tts(X,Y,test_size=0.1,random_state=42)

In [86]:
'''Define entropy and information gain'''
def entropy(y):
    classes, counts=np.unique(y,return_counts=True)
    probabilities=counts/counts.sum()
    return -np.sum(probabilities*np.log2(probabilities))
def info_gain(y,y_left,y_right):
    n=len(y)
    nl,nr=len(y_left),len(y_right)
    if nl==0 or nr==0:
        return 0.0
    return entropy(y) - (nl/n)*entropy(y_left) - (nr/n)*entropy(y_right)

In [87]:
'''Best Split function'''
def best_split(X,y):
    best_gain=-1
    best_feature,best_threshold=None,None
    n_samples,n_features=X.shape
    for feature in range(n_features):
        thresholds=np.unique(X[:,feature])
        for threshold in thresholds:
            left=y[X[:,feature]<=threshold]
            right=y[X[:,feature]>threshold]
            if len(left)==0 or len(right)==0:
                continue
            gain=info_gain(y,left,right)
            if gain>best_gain:
                best_gain=gain
                best_feature=feature
                best_threshold=threshold
    return best_feature,best_threshold

In [88]:
'''Build the tree'''
class Node:
    def __init__(self,feature=None,threshold=None,left=None,right=None,value=None):
        self.feature=feature
        self.threshold=threshold
        self.left=left
        self.right=right
        self.value=value
def build_tree(X,y,depth=0,max_depth=3):
    if(len(set(y))==1 or depth==max_depth):
        leaf_value=np.argmax(np.bincount(y))
        return Node(value=leaf_value)
    feature,threshold=best_split(X,y)
    if feature is None:
        leaf_value=np.argmax(np.bincount(y))
        return Node(value=leaf_value)
    lidx=X[:,feature]<=threshold
    ridx=X[:,feature]>threshold
    left=build_tree(X[lidx],y[lidx],depth+1)
    right=build_tree(X[ridx],y[ridx],depth+1)
    return Node(feature,threshold,left,right)

In [89]:
'''Predict function'''
def pred(node,x):
    while node.value is None:
        if x[node.feature]<=node.threshold:
            node=node.left
        else:
            node=node.right
    return node.value
def predict(tree,X):
    return np.array([pred(tree,x) for x in X])

In [90]:
'''Training and Evaluation'''
tree=build_tree(x_train,y_train)
y_pred=predict(tree,x_test)
accuracy=np.mean(y_pred==y_test)*100
print("Accuracy is : ",accuracy)

Accuracy is :  82.22222222222221
